In [ ]:
# ========================================
# CELL 1 — Fix Dependencies + Install Libraries
# ========================================

# Fix numpy compatibility
!pip install numpy==1.26.4 --quiet

# Install PyTorch Compatible Version
!pip install torch torchvision torchaudio --quiet

# Install Required Libraries
!pip install transformers --quiet
!pip install datasets --quiet
!pip install accelerate --quiet
!pip install librosa --quiet
!pip install soundfile --quiet
!pip install openai-whisper --quiet
!pip install pyannote.audio --quiet
!pip install sentencepiece --quiet
!pip install huggingface_hub --quiet
!pip install audiomentations --quiet
!pip install scipy --quiet
!pip install pandas --quiet

print("✅ Environment Ready Successfully")

In [ ]:
# ========================================
# FIX CELL — Torch Compatibility
# ========================================

!pip install torch torchvision torchaudio --upgrade --quiet

print("✅ Torch compatibility fixed")

In [1]:
# ========================================
# CELL 2 — Extract ZIP + Input Audio
# ========================================

import os
import zipfile

# Create folder
os.makedirs("audio_data", exist_ok=True)

# ZIP path
zip_path = "a.zip"

# Extract ZIP
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("audio_data")

print("✅ ZIP Extracted Successfully")


# This is the MAIN INPUT for entire model
audio_path = "audio_data/b.mpeg"

print("Model Input Audio:", audio_path)

FileNotFoundError: [Errno 2] No such file or directory: 'a.zip'

In [10]:
# ========================================
# CELL 3 — Universal Audio Loader
# ========================================

import librosa
import soundfile as sf
import os

# Load audio
audio, sr = librosa.load(audio_path, sr=16000)

print("Audio Loaded Successfully")
print("Sample Rate:", sr)
print("Audio Length:", len(audio)/sr, "seconds")

# Convert to WAV (Universal format for models)
converted_audio_path = "processed_audio.wav"

sf.write(converted_audio_path, audio, sr)

print("Converted Audio Saved:", converted_audio_path)

Audio Loaded Successfully
Sample Rate: 16000
Audio Length: 51.46125 seconds
Converted Audio Saved: processed_audio.wav


In [11]:
# ========================================
# CELL 4 — Speech Recognition (Whisper)
# ========================================

import whisper

# Load Whisper Model
model = whisper.load_model("base")

# Transcribe Audio
result = model.transcribe(converted_audio_path)

# Extract text
transcription = result["text"]

print("Transcription:")
print(transcription)

/nlsasfs/home/aikosh/prod-aikosh34/.venv/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription:
 No! Rose, now's your chance. Go live a normal life. Make friends. No one will ever call you a freak again. When? Rose, just go! Go! Go! Go! Go! Go! Go! I love you. But I need to know you're safe. So please, just... Aint that you're in for a bad... We got this. Get out of here! No! No! No! There you go, sweetheart. Don't you worry. I'll be right downstairs. Daddy will let those weird fairy-tell monsters get you.


In [12]:
# ========================================
# CELL 5 — Emotion Detection (Wav2Vec2)
# ========================================

from transformers import pipeline

# Load Emotion Model
emotion_classifier = pipeline(
    "audio-classification",
    model="superb/wav2vec2-base-superb-er"
)

# Predict Emotion
emotion = emotion_classifier(converted_audio_path)

print("Detected Emotion:")
print(emotion)

/nlsasfs/home/aikosh/prod-aikosh34/.venv/lib/python3.10/site-packages/transformers/configuration_utils.py:306: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


Detected Emotion:
[{'score': 0.5894801616668701, 'label': 'sad'}, {'score': 0.350097119808197, 'label': 'hap'}, {'score': 0.05848982185125351, 'label': 'neu'}, {'score': 0.0019329359056428075, 'label': 'ang'}]


In [13]:
# ========================================
# CELL 6 — Non-Speech Detection (AST)
# ========================================

from transformers import pipeline

# Load AST model
audio_classifier = pipeline(
    "audio-classification",
    model="MIT/ast-finetuned-audioset-10-10-0.4593"
)

# Detect non-speech audio
non_speech = audio_classifier(converted_audio_path)

print("Non-Speech Audio Detection:")
print(non_speech[:5])

Non-Speech Audio Detection:
[{'score': 0.5220869183540344, 'label': 'Speech'}, {'score': 0.38930222392082214, 'label': 'Music'}, {'score': 0.010108716785907745, 'label': 'Female speech, woman speaking'}, {'score': 0.009419901296496391, 'label': 'Conversation'}, {'score': 0.007541874423623085, 'label': 'Male speech, man speaking'}]


In [25]:
# ========================================
# CELL 7 — Mistral 7B 4-bit Reasoning
# ========================================

!pip install transformers accelerate bitsandbytes --quiet

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")

# Load 4-bit model
model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.float16
)

prompt = f"""
You are an Audio Language Model.

Audio Transcription:
{transcription}

Emotion:
{emotion}

Non Speech:
{non_speech}

Provide:
1. Number of speakers
2. Conversation summary
3. Audio scene description
4. Final reasoning
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.7
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\nFinal Audio Reasoning:\n")
print(response)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.


RuntimeError: Failed to import transformers.integrations.bitsandbytes because of the following error (look up to see its traceback):
No module named 'triton._C.libtriton.triton'; 'triton._C.libtriton' is not a package

In [26]:
# ========================================
# Install CPU Quantized Mistral
# ========================================

!pip install llama-cpp-python --quiet
!pip install huggingface_hub --quiet

print("✅ Mistral GGUF ready")

✅ Mistral GGUF ready


In [27]:
from llama_cpp import Llama

model = Llama.from_pretrained(
    repo_id="TheBloke/Mistral-7B-Instruct-v0.2-GGUF",
    filename="mistral-7b-instruct-v0.2.Q4_K_M.gguf",
    n_ctx=4096
)

prompt = f"""
Audio Transcription:
{transcription}

Emotion:
{emotion}

Non Speech:
{non_speech}

Give:
1. Number of speakers
2. Conversation summary
3. Audio scene
4. Final reasoning
"""

output = model(prompt, max_tokens=300)

print("\nFinal Reasoning:\n")
print(output["choices"][0]["text"])

mistral-7b-instruct-v0.2.Q4_K_M.gguf:   0%|          | 0.00/4.37G [00:00<?, ?B/s]

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /nlsasfs/home/aikosh/prod-aikosh34/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/./mistral-7b-instruct-v0.2.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32      


Final Reasoning:

5. Emotion analysis

1. Number of speakers:
There are two speakers in this audio clip. The first speaker is a woman, and the second speaker is a man.

2. Conversation summary:
The woman, named Rose, is having a conversation with her father about leaving their home. Her father is urging her to leave and live a normal life, while Rose is hesitant. He assures her that he will be there to protect her, implying that there are "weird fairy-tell monsters" that may harm her.

3. Audio scene:
The audio scene is a conversation between a father and his daughter. The father is trying to persuade the daughter to leave their home and live a normal life. The daughter is reluctant, and the father reassures her that he will protect her from harm.

4. Final reasoning:
Based on the conversation, it seems that the father and daughter are in some kind of danger or living in an unusual circumstance. The father's insistence that the daughter leave and his reference to "weird fairy-tell mon